# Phase 4: Double Machine Learning (DML) Price Elasticity Estimation

In this notebook, we implement Double Machine Learning (DML) using Microsoft's `econml` library to estimate the true causal price elasticity of demand. DML allows us to control for high-dimensional store, product, seasonal, and promotional confounders non-parametrically using machine learning nuisance models. By applying Neyman orthogonality and cross-fitting, we obtain unbiased estimates and confidence intervals for the Average Treatment Effect (ATE).

## 1. Setup and Libraries

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.formula.api as smf
from sklearn.ensemble import HistGradientBoostingRegressor, RandomForestRegressor
from sklearn.model_selection import cross_val_score, cross_val_predict
from econml.dml import LinearDML
import os

# Visualization Settings
%matplotlib inline
sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (12, 6)
plt.rcParams["font.size"] = 12

# Load clean dataset
CLEAN_DATA_PATH = '../data/processed/cereales_features.csv'
print("Loading dataset...")
df = pd.read_csv(CLEAN_DATA_PATH, parse_dates=['START_DATE', 'END_DATE'])
print(f"Data loaded. Shape: {df.shape}")

Loading dataset...


/var/folders/bv/7hq66ftx4dv55527kwllf2340000gn/T/ipykernel_60163/519675297.py:20: DtypeWarning: Columns (15) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(CLEAN_DATA_PATH, parse_dates=['START_DATE', 'END_DATE'])


Data loaded. Shape: (4638752, 46)


### 1.1 Representative Sampling for Computational Efficiency

Estimating a DML model with a 5-fold cross-fitting scheme on 4.7 million rows using tree-based regressors is computationally demanding and can lead to memory exhaustion or excessively long training times. 

To ensure efficient execution without sacrificing statistical power, we draw a representative random sample of **100,000 observations**. In causal inference on large datasets, a sample of this size is more than sufficient to yield highly precise point estimates and very narrow confidence intervals.

In [2]:
df_sample = df.sample(n=100000, random_state=42).copy()
print(f"Sample size selected: {df_sample.shape[0]} observations.")

Sample size selected: 100000 observations.


## 2. Matrix Preparation (Y, T, W, X)

We construct the four matrices required for the DML pipeline:
*   **Outcome ($Y$)**: `log_units` (logarithm of weekly units sold).
*   **Treatment ($T$)**: `log_price` (logarithm of price, continuous treatment: `discrete_treatment=False`).
*   **Confounders ($W$)**: Variables that simultaneously affect both treatment (pricing decisions) and outcome (sales). This includes `promotion` status, price log difference lag (`lag_price_change` to prevent weak overlap/overcontrolling without leaking treatment), lagged demand (`log_lag_units`), competitor pricing (`log_competitor_price`), specific holiday indicators (`holiday_thanksgiving`, `holiday_christmas`, `back_to_school`), continuous time trend (`time_trend`), month x price zone interaction dummies, brand dummies, and store fixed effects (`STORE` as categorical) to capture baseline time-invariant differences. Dummies are one-hot encoded with `drop_first=True` to prevent multicollinearity.
*   **Heterogeneity ($X$)**: Store price zones (`PRICE_ZONE` one-hot encoded). We isolate this variable to moderate the treatment effect, establishing the foundation for store-specific analysis in the next phase.

*Note on overlap prevention: `PRICE_ZONE` is time-invariant for each store. Therefore, `STORE` dummies (in $W$) completely span the price zone. However, we keep `PRICE_ZONE` in $X$ and `STORE` in $W$ because $W$ acts as the nuisance projection matrix (controlling for individual store differences), while $X$ serves as the moderation space to check CATE across pricing zones.*

In [3]:
# Outcome and Treatment vectors
Y = df_sample['log_units'].values
T = df_sample['log_price'].values

# Confounder matrix W using lag_price_change to avoid treatment leakage
confounder_cols = ['promotion', 'time_trend', 'log_competitor_price', 'lag_price_change', 'log_lag_units', 
                   'holiday_thanksgiving', 'holiday_christmas', 'back_to_school', 'month_zone', 'brand', 'STORE']
W_df = pd.get_dummies(df_sample[confounder_cols], columns=['month_zone', 'brand', 'STORE'], drop_first=True, dtype=float)
W = W_df.values

# Heterogeneity matrix X
X_df = pd.get_dummies(df_sample[['PRICE_ZONE']], columns=['PRICE_ZONE'], drop_first=True, dtype=float)
X = X_df.values

print(f"Matrix dimensions:")
print(f"Y: {Y.shape}")
print(f"T: {T.shape}")
print(f"W (Confounders): {W.shape} columns: {W_df.columns[:5].tolist()} ...")
print(f"X (Heterogeneity): {X.shape} columns: {X_df.columns.tolist()}")

Matrix dimensions:
Y: (100000,)
T: (100000,)
W (Confounders): (100000, 158) columns: ['promotion', 'time_trend', 'log_competitor_price', 'lag_price_change', 'log_lag_units'] ...
X (Heterogeneity): (100000, 3) columns: ['PRICE_ZONE_High', 'PRICE_ZONE_Low', 'PRICE_ZONE_Medium']


## 3. Double Machine Learning (DML) Modeling

We select `HistGradientBoostingRegressor` as the nuisance estimators for both $\text{model}_y$ (predicting demand from confounders) and $\text{model}_t$ (predicting prices from confounders). HistGradientBoosting is selected over Random Forests because it is significantly faster on large datasets and natively handles complex non-linear interactions (e.g., temporal demand shifts and promotional overlaps) that simple linear projections or slow random forests struggle with.

We use `LinearDML` from `econml` with a 5-fold cross-fitting scheme (`cv=5`) to prevent overfitting bias (Neyman orthogonality).

In [4]:
# Define nuisance models
model_y = HistGradientBoostingRegressor(max_iter=50, random_state=42)
model_t = HistGradientBoostingRegressor(max_iter=50, random_state=42)

# Instantiate DML Estimator
est = LinearDML(
    model_y=model_y,
    model_t=model_t,
    discrete_treatment=False,
    cv=5,
    random_state=42
)

# Fit DML on the sample
print("Fitting DML estimator (this might take around 30-60 seconds due to 5-fold cross-fitting)... ")
est.fit(Y, T, X=X, W=W)
print("DML fitting completed successfully.")

Fitting DML estimator (this might take around 30-60 seconds due to 5-fold cross-fitting)... 


DML fitting completed successfully.


/Users/percy/Proyectos/Pricing_Elasticity/venv/lib/python3.9/site-packages/econml/sklearn_extensions/linear_model.py:1846: RuntimeWarning: divide by zero encountered in matmul
  var_i = sample_var + (y - np.matmul(X, param))**2
/Users/percy/Proyectos/Pricing_Elasticity/venv/lib/python3.9/site-packages/econml/sklearn_extensions/linear_model.py:1846: RuntimeWarning: overflow encountered in matmul
  var_i = sample_var + (y - np.matmul(X, param))**2
/Users/percy/Proyectos/Pricing_Elasticity/venv/lib/python3.9/site-packages/econml/sklearn_extensions/linear_model.py:1846: RuntimeWarning: invalid value encountered in matmul
  var_i = sample_var + (y - np.matmul(X, param))**2
/Users/percy/Proyectos/Pricing_Elasticity/venv/lib/python3.9/site-packages/econml/sklearn_extensions/linear_model.py:1867: RuntimeWarning: divide by zero encountered in matmul
  weighted_sigma = np.matmul(WX.T, WX * var_i.reshape(-1, 1))
/Users/percy/Proyectos/Pricing_Elasticity/venv/lib/python3.9/site-packages/econml/skl

## 4. Estimation of the Average Treatment Effect (ATE)

We extract the Average Treatment Effect (ATE) and its 95% confidence interval using the fitted model.

In [5]:
# Calculate ATE and its 95% Confidence Interval
ate_est = est.ate(X)
ate_ci = est.ate_interval(X, alpha=0.05)

print(f"Estimated Causal Price Elasticity (DML ATE): {ate_est:.4f}")
print(f"95% Confidence Interval: [{ate_ci[0]:.4f}, {ate_ci[1]:.4f}]")

Estimated Causal Price Elasticity (DML ATE): -0.0884
95% Confidence Interval: [-0.1058, -0.0709]


/Users/percy/Proyectos/Pricing_Elasticity/venv/lib/python3.9/site-packages/econml/sklearn_extensions/linear_model.py:1519: RuntimeWarning: divide by zero encountered in matmul
  return np.matmul(X, self._param)
/Users/percy/Proyectos/Pricing_Elasticity/venv/lib/python3.9/site-packages/econml/sklearn_extensions/linear_model.py:1519: RuntimeWarning: overflow encountered in matmul
  return np.matmul(X, self._param)
/Users/percy/Proyectos/Pricing_Elasticity/venv/lib/python3.9/site-packages/econml/sklearn_extensions/linear_model.py:1519: RuntimeWarning: invalid value encountered in matmul
  return np.matmul(X, self._param)
/Users/percy/Proyectos/Pricing_Elasticity/venv/lib/python3.9/site-packages/econml/sklearn_extensions/linear_model.py:1519: RuntimeWarning: divide by zero encountered in matmul
  return np.matmul(X, self._param)
/Users/percy/Proyectos/Pricing_Elasticity/venv/lib/python3.9/site-packages/econml/sklearn_extensions/linear_model.py:1519: RuntimeWarning: overflow encountered in 

### 4.1 Comparative Analysis: Naive OLS vs. DML

Let's fit a naive OLS model on the same 100k sample to construct a direct comparison table.

In [6]:
# Fit OLS on the sample
ols_sample = smf.ols("log_units ~ log_price", data=df_sample).fit()
ols_coef = ols_sample.params['log_price']
ols_ci = ols_sample.conf_int().loc['log_price'].values

# Construct comparison table
comparison_df = pd.DataFrame({
    'Model': ['Naive OLS (Biased)', 'Double Machine Learning (Causal)'],
    'Elasticity': [ols_coef, ate_est],
    '95% CI Lower': [ols_ci[0], ate_ci[0]],
    '95% CI Upper': [ols_ci[1], ate_ci[1]]
})
display(comparison_df)

,Model,Elasticity,95% CI Lower,95% CI Upper
0,Naive OLS (Biased),-0.313658,-0.333016,-0.294300
1,Double Machine Learning (Causal),-0.088364,-0.105832,-0.070896


### 4.2 Economic Interpretation of the Comparative Table

*   **Naive OLS Elasticity**: **-0.3390** (95% CI: `[-0.3586, -0.3195]`). This estimate suggests that demand is inelastic. This moderate price sensitivity is biased by the inclusion of promotional spikes.
*   **DML Causal Elasticity**: **-0.0884** (95% CI: `[-0.1058, -0.0709]`). When we control for promotions, competitor prices, and lagged dynamics using a corrected confounder specification, the pure price elasticity (excluding promotional lift/advertising) collapses closer to zero, indicating that baseline demand is extremely inelastic.
*   **Statistical Significance vs. Economic Significance**: Due to our large sample size of 100,000 observations, our DML estimate has an extremely narrow confidence interval (~0.035 wide), making the elasticity of **-0.0884** highly statistically significant ($p < 0.001$). However, we must distinguish this from *economic significance*: an elasticity of -0.09 is economically small. This tells us that changing the baseline shelf price in isolation, without active in-store displays or flyer promotions, has a negligible effect on driving sales volumes.
*   **Why the shift?** OLS is biased more negative because it ignores promotional demand shocks and competitor price movements. DML separates the two, isolating the baseline shelf elasticity.

## 5. Robustness Checks and Diagnostics

To validate our causal pipeline and meet rigorous econometric standards, we implement three diagnostic checks.

### 5.1 Placebo Test (Treatment Permutation)

We randomly permute the pricing treatment vector `T` to break the causal relationship with units sold. Running DML on this placebo treatment should result in a causal effect that collapses to zero.

In [7]:
# Permute treatment randomly
np.random.seed(42)
T_placebo = np.random.permutation(T)

# Fit Placebo DML
est_placebo = LinearDML(
    model_y=model_y,
    model_t=model_t,
    discrete_treatment=False,
    cv=5,
    random_state=42
)

print("Running Placebo DML...")
est_placebo.fit(Y, T_placebo, X=X, W=W)
ate_placebo = est_placebo.ate(X)
ci_placebo = est_placebo.ate_interval(X, alpha=0.05)

print(f"Placebo Causal Elasticity: {ate_placebo:.4f} (95% CI: [{ci_placebo[0]:.4f}, {ci_placebo[1]:.4f}])")

Running Placebo DML...


Placebo Causal Elasticity: -0.0120 (95% CI: [-0.0251, 0.0011])


/Users/percy/Proyectos/Pricing_Elasticity/venv/lib/python3.9/site-packages/econml/sklearn_extensions/linear_model.py:1846: RuntimeWarning: divide by zero encountered in matmul
  var_i = sample_var + (y - np.matmul(X, param))**2
/Users/percy/Proyectos/Pricing_Elasticity/venv/lib/python3.9/site-packages/econml/sklearn_extensions/linear_model.py:1846: RuntimeWarning: overflow encountered in matmul
  var_i = sample_var + (y - np.matmul(X, param))**2
/Users/percy/Proyectos/Pricing_Elasticity/venv/lib/python3.9/site-packages/econml/sklearn_extensions/linear_model.py:1846: RuntimeWarning: invalid value encountered in matmul
  var_i = sample_var + (y - np.matmul(X, param))**2
/Users/percy/Proyectos/Pricing_Elasticity/venv/lib/python3.9/site-packages/econml/sklearn_extensions/linear_model.py:1867: RuntimeWarning: divide by zero encountered in matmul
  weighted_sigma = np.matmul(WX.T, WX * var_i.reshape(-1, 1))
/Users/percy/Proyectos/Pricing_Elasticity/venv/lib/python3.9/site-packages/econml/skl

### 5.2 Sensitivity Analysis: Nuisance Model Stability

We replace `HistGradientBoostingRegressor` with a `RandomForestRegressor` (deliberately configured with a simple structure: `n_estimators=10` and `max_depth=5`) to assess the sensitivity of the ATE to the choice of nuisance models.

*Methodological Note*: A Random Forest with only 10 trees and a depth of 5 is a much simpler, less flexible model than the Hist Gradient Boosting used in the primary specification. Showing that the estimated causal price elasticity remains highly stable even when using a significantly limited nuisance model is a very strong argument for the robustness of our causal pipeline: it demonstrates that the core causal finding does not depend on hyper-sophisticated modeling architectures.

In [8]:
# Define Random Forest nuisance models (fast configurations)
model_y_rf = RandomForestRegressor(n_estimators=10, max_depth=5, random_state=42, n_jobs=-1)
model_t_rf = RandomForestRegressor(n_estimators=10, max_depth=5, random_state=42, n_jobs=-1)

est_rf = LinearDML(
    model_y=model_y_rf,
    model_t=model_t_rf,
    discrete_treatment=False,
    cv=5,
    random_state=42
)

print("Running RF-DML Sensitivity Test...")
est_rf.fit(Y, T, X=X, W=W)
ate_rf = est_rf.ate(X)
ci_rf = est_rf.ate_interval(X, alpha=0.05)

print(f"Random Forest Elasticity: {ate_rf:.4f} (95% CI: [{ci_rf[0]:.4f}, {ci_rf[1]:.4f}])")

Running RF-DML Sensitivity Test...


Random Forest Elasticity: -0.0867 (95% CI: [-0.1041, -0.0693])


/Users/percy/Proyectos/Pricing_Elasticity/venv/lib/python3.9/site-packages/econml/sklearn_extensions/linear_model.py:1846: RuntimeWarning: divide by zero encountered in matmul
  var_i = sample_var + (y - np.matmul(X, param))**2
/Users/percy/Proyectos/Pricing_Elasticity/venv/lib/python3.9/site-packages/econml/sklearn_extensions/linear_model.py:1846: RuntimeWarning: overflow encountered in matmul
  var_i = sample_var + (y - np.matmul(X, param))**2
/Users/percy/Proyectos/Pricing_Elasticity/venv/lib/python3.9/site-packages/econml/sklearn_extensions/linear_model.py:1846: RuntimeWarning: invalid value encountered in matmul
  var_i = sample_var + (y - np.matmul(X, param))**2
/Users/percy/Proyectos/Pricing_Elasticity/venv/lib/python3.9/site-packages/econml/sklearn_extensions/linear_model.py:1867: RuntimeWarning: divide by zero encountered in matmul
  weighted_sigma = np.matmul(WX.T, WX * var_i.reshape(-1, 1))
/Users/percy/Proyectos/Pricing_Elasticity/venv/lib/python3.9/site-packages/econml/skl

### 5.3 Quality Diagnostics of Nuisance Models

The validity of DML relies on the predictive performance of the nuisance models ($R^2$ in cross-validation). If models cannot predict price and sales reasonably well from $W$, the residuals will contain structural noise that biases the causal parameters.

In [9]:
# Run cross-validation on nuisance models
cv_scores_y = cross_val_score(model_y, W, Y, cv=3, scoring='r2')
cv_scores_t = cross_val_score(model_t, W, T, cv=3, scoring='r2')

print(f"Nuisance Model Y (Demand) mean R^2: {cv_scores_y.mean():.4f} (Folds: {cv_scores_y})")
print(f"Nuisance Model T (Price) mean R^2: {cv_scores_t.mean():.4f} (Folds: {cv_scores_t})")

Nuisance Model Y (Demand) mean R^2: 0.5442 (Folds: [0.548819   0.53336834 0.55032372])
Nuisance Model T (Price) mean R^2: 0.1772 (Folds: [0.17704492 0.17722073 0.17720907])


### 5.4 Treatment Residual Variance Diagnostic (Overcontrolling Check)

We evaluate the remaining variance of the price treatment residuals ($T_{res} = T - \hat{T}$). If the price nuisance model predicts pricing decisions too well ($R^2 \approx 100\%$), the residual variance collapses to zero. This represents an "overcontrolling" scenario that violates the causal positivity/overlap assumption, blowing up our standard errors. A healthy ratio of residual-to-original variance confirms that we retain enough independent variation in price to identify the causal elasticity parameter.

In [10]:
T_pred = cross_val_predict(model_t, W, T, cv=5)
T_res = T - T_pred

print(f"Original Treatment (T) Variance: {T.var():.4f}")
print(f"Residual Treatment (T_res) Variance: {T_res.var():.4f}")
print(f"Ratio (Residual/Original Variance): {T_res.var() / T.var():.4f}")

Original Treatment (T) Variance: 0.0839
Residual Treatment (T_res) Variance: 0.0689
Ratio (Residual/Original Variance): 0.8217


## 6. Identification Assumption: Conditional Unconfoundedness

**Theoretical Disclaimer**: 
The causal interpretation of our DML elasticity assumes **Conditional Unconfoundedness (Selection on Observables)**: 
$$T \perp Y(t) \mid (W, X)$$
This states that conditional on the store, brand, promotion, month, and year dummies in our confounder matrix, there are no omitted variables that simultaneously cause price changes and sales volume fluctuations.

While our confounder matrix is comprehensive, potential sources of residual bias include **local advertising (e.g., local mailers, shelf-talkers)** and **competitor pricing strategies** that are not fully captured by store-brand dummies. We should acknowledge this limitation when reporting these results to business stakeholders.

## 7. Appendix: What is Weak Overlap?
 
**Overlap** is a core assumption in causal inference: even after controlling for confounders, treatment still needs to vary "freely" enough to be compared across similar units. **Weak overlap** happens when the confounders predict the treatment almost perfectly, leaving too little independent variation to identify a stable causal effect.
 
In DML, this matters because the effect is estimated from the residual variation in treatment after removing what confounders explain. If that residual variation is tiny, the estimate becomes highly sensitive to noise and to the choice of nuisance model — small errors get amplified.
 
**What we saw:** an early version of $W$ included `log_lag_price` (last week's price level). Since prices are highly autocorrelated week to week, this let `model_t` predict price almost perfectly ($R^2 = 0.93$), leaving only 7.4% of price variance to identify the effect. The result was an implausible ATE (-2.48) that swung sharply between nuisance models (HGB vs. RF).
 
**Fix:** replacing the price level with `lag_price_change` (the pre-treatment price *change*, week t-1 vs. t-2) kept the useful signal about pricing dynamics without absorbing the current price level. This brought $R^2$ down to 0.18 and the residual variance ratio up to 0.82 — healthy overlap — and produced a stable ATE (-0.088) consistent across nuisance models (HGB: -0.088, RF: -0.087).
 
**Note:** this is different from the "bad controls" issue discussed earlier (e.g. `STORE`/`UPC` fixed effects). Bad controls bias the effect in a direction; weak overlap doesn't necessarily bias it, but makes it unstable and unreliable. Both come from confounder specification, but call for different diagnostics: watch $R^2$ of `model_t` (red flag above ~0.7–0.8) and the residual variance ratio of T (`T_res.var() / T.var()`, red flag below ~0.2).

## 8. Conclusion and Next Steps

Our DML model successfully isolated the causal price elasticity of **-0.0884** (with a narrow 95% CI: `[-0.1058, -0.0709]`). The placebo test collapsed to ~0, confirming that our causal estimate is not a spurious artifact of the pipeline. 

This establishes a robust Average Treatment Effect (ATE). In the next phase (`notebooks/05_causal_forest_cate.ipynb`), we will expand this formulation to estimate the **Conditional Average Treatment Effect (CATE)** across individual pricing zones using **Causal Forest DML**, enabling localized pricing optimization.